In [1]:
import originpro as op
import numpy as np
import pandas as pd

import os
# Very useful, especially during development, when you are
# liable to have a few uncaught exceptions.
# Ensures that the Origin instance gets shut down properly.
# Note: only applicable to external Python.
import sys
def origin_shutdown_exception_hook(exctype, value, traceback):
    '''Ensures Origin gets shut down if an uncaught exception'''
    op.exit()
    sys.__excepthook__(exctype, value, traceback)
if op and op.oext:
    sys.excepthook = origin_shutdown_exception_hook


# Set Origin instance visibility.
# Important for only external Python.
# Should not be used with embedded Python.
if op.oext:
    op.set_show(True)


# Example of opening a project and reading data.

# We'll open the Tutorial Data.opju project that ships with Origin.
src_opju = r"C:\usrspace\mywork\edges4.opju"   # <— 用绝对路径；注意路径和文件名是否正确
print("File exists? ", os.path.exists(src_opju))
ok = op.open(file=src_opju)
print("Project opened? ", ok)
import draw.pymatlab2.origin_function.csv2origin as csv2origin

import draw.pymatlab2.origin_function.fillgap as fillgap

File exists?  True
Project opened?  True


In [2]:
from pathlib import Path
import draw.pymatlab2.origin_function.csv2origin as csv2origin
from config import DATA_DIR,INPUT_DIR
time_2_build = 40
TIME_2_BUILD = time_2_build
from pathlib import Path
# 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
VERSION = os.getenv("TOPOLOGY_VERSION", f"topology_{TIME_2_BUILD}")



FIGURE_DIR    = Path(INPUT_DIR) / VERSION / "figure"

In [2]:

# ===== 2) 列出所有页面类型，看看有没有工作簿 =====
wbooks = list(op.pages('w'))   # 所有工作簿（Worksheet Book）
mbks   = list(op.pages('m'))   # 所有矩阵簿（Matrix Book）
graphs = list(op.pages('g'))   # 所有图页（Graph），仅排查
notes  = list(op.pages('n'))   # 所有Notes，仅排查

print("#Workbooks:", len(wbooks), "| #MatrixBooks:", len(mbks), "| #Graphs:", len(graphs), "| #Notes:", len(notes))

# ===== 3) 如果有工作簿：打印每个工作簿下所有工作表名 =====
for wb in wbooks:
    sheet_names = [wks.name for wks in wb]
    print(f"[{wb.name}] -> {sheet_names}")

# ===== 4) 如果是矩阵簿：打印矩阵表名（有些项目只有矩阵簿，没有工作簿）=====
for mb in mbks:
    ms_names = [ms.name for ms in mb]
    print(f"[{mb.name}] (Matrix) -> {ms_names}")

# ===== 5) 若你想强行拿到第一个工作簿并遍历 =====
if wbooks:
    wb = wbooks[0]
    print("Active to:", wb.name)
    wb.activate()  # 让它成为激活窗口
    for wks in wb:
        print("Sheet:", wks.name)

#Workbooks: 1 | #MatrixBooks: 0 | #Graphs: 10 | #Notes: 0
[Book1] -> ['pending_edges_ttb10', 'pending_edges_ttb20', 'pending_edges_ttb30', 'pending_edges_ttb40', 'pending_edges_ttb50']
Active to: Book1
Sheet: pending_edges_ttb10
Sheet: pending_edges_ttb20
Sheet: pending_edges_ttb30
Sheet: pending_edges_ttb40
Sheet: pending_edges_ttb50


In [4]:

filename = f"avgspath_g0_4_ttb{TIME_2_BUILD}"
FIGURE_DIR = Path(FIGURE_DIR)           # 确保是 Path
csv1_path = FIGURE_DIR / f"{filename}.csv"

csv2origin.csv_to_origin(csv1_path, book="Book2", sheet=filename, clear=True)

WSheet: [Book2]avgspath_g0_4_ttb40

In [54]:
#when we finished the write to the origin ,we first need fill the gap



# mod_gap.py
from __future__ import annotations
import pandas as pd
from typing import Iterable, Optional, Sequence
import draw.pymatlab2.origin_function.csv2origin as csv2origin

import draw.pymatlab2.origin_function.fillgap as fillgap


fillgap.origin_fill_time_gaps(
    book="Book1",
    sheet=filename,     # 原表名
    time_start=0,
    step=1,
    fill_value=0,
    dest_sheet=None,    # None=覆盖写回原表；也可以 "xxx_filled"
    clear_dest=True
)

,time,pending_edges
0,0,41
1,1,41
2,2,41
3,3,41
4,4,41
...,...,...
21889,21889,21
21890,21890,21
21891,21891,21
21892,21892,21


In [6]:
TTB_VALUES = [  10,20,30,40,50,60, 70, 80,90, 100, 110, 120,130,140]

for ttb in TTB_VALUES:
    VERSION = os.getenv("TOPOLOGY_VERSION", f"topology_{ttb}")
    FIGURE_DIR    = Path(INPUT_DIR) / VERSION / "figure"

    filename = f"avgspath_g0_4_ttb{ttb}"


    csv1_path = FIGURE_DIR / f"{filename}.csv"

    csv2origin.csv_to_origin(csv1_path, book="Book2", sheet=filename, clear=True)
#     fillgap.origin_fill_time_gaps(
#     book="Book2",
#     sheet=filename,     # 原表名
#     time_start=0,
#     step=1,
#     fill_value=0,
#     dest_sheet=None,    # None=覆盖写回原表；也可以 "xxx_filled"
#     clear_dest=True
# )


In [7]:
import draw.pymatlab2.origin_function.readorigin as readorigin
TARGET_BOOK = 'Book2'
ALL_INTER_SHEET   = 'avgspath_g0_4_ttb10'   # 结果写到这个表

ws = readorigin.get_ws(TARGET_BOOK, ALL_INTER_SHEET)
df = readorigin.ws_to_df(ws)

In [10]:
qujian  = []



mask = df['step'].notna()  # 只保留有时间的行，保持对齐
t_arr = pd.to_numeric(df.loc[mask, 'step'], errors='coerce').astype(int).to_numpy()
e_arr = pd.to_numeric(df.loc[mask, 'avg_shortest_path'], errors='coerce').fillna(0).astype(int).to_numpy()



In [11]:
sum = 0
for t, e in zip(t_arr, e_arr):
    # t 是 int 时间，e 是 int 的 pending_edges（NaN 已按 0 处理）
    # 在这里写你的逻辑
    sum = sum+e
print(sum/22005)


12.170824812542604


接下来的内容是,我们要计算,平均建立链路条数




In [2]:
import draw.pymatlab2.origin_function.write2origin as write2origin
import re
TARGET_BOOK = 'Book2'
OUT_SHEET   = 'avgspath_all '   # 结果写到这个表

bk = op.find_book('w', TARGET_BOOK)
if bk is None:
    raise RuntimeError(f"找不到工作簿 '{TARGET_BOOK}'")

results = []


for wks in bk:
    nm = wks.name
    if not nm.startswith('avgspath_g0_4_ttb'):
        continue

    # ===== 读取（Long Name 为表头）=====
    df = wks.to_df(head='L')
    if not {'step','avg_shortest_path'}.issubset(df.columns):
        print(f"跳过（缺列）: [{TARGET_BOOK}] {nm}")
        continue

    # ===== 规范化 =====
    df = df[['step','avg_shortest_path']].copy()
    df['step'] = pd.to_numeric(df['step'], errors='coerce')
    df['avg_shortest_path'] = pd.to_numeric(df['avg_shortest_path'], errors='coerce')
    df = df.dropna(subset=['step']).copy()
    df['step'] = df['step'].astype(int)
    df = df.sort_values('step')

    if df.empty:
        print(f"跳过（空表）: [{TARGET_BOOK}] {nm}")
        continue

    # ===== 平均切换条数（每秒平均）=====
    observed_seconds = int(len(df))                        # T
    pe = df['avg_shortest_path'].fillna(0)                     # L_i（NaN 记为 0）
    sum_link = float(pe.sum())                             # Σ L_i
    avg_link = sum_link / observed_seconds if observed_seconds > 0 else 0.0  # ΣL_i / T

    # === 新增：相对波动（CV） ===
    series = df['avg_shortest_path'].astype(float).dropna()
    mean_val = float(series.mean())
    std_val  = float(series.std(ddof=1)) if len(series) > 1 else 0.0
    cv = (std_val / mean_val) if mean_val != 0 else float('nan')

    results.append({
        'book': TARGET_BOOK,
        'sheet': nm,
        'avg_hops': avg_link,
            'std_val': std_val,                     # ← 新增


    })

# ===== 汇总并写回 Origin =====
if results:
    df_sum = pd.DataFrame(results)

    # 从 sheet 名里提取 ttb 数字用于排序（可选）
    # def _pick_ttb(s):
    #     try:
    #         return int(''.join(ch for ch in s if ch.isdigit()))
    #     except Exception:
    #         return None
    def _pick_ttb(s: str):
    # 只取 ttb 后面连续的数字，忽略前面的 g0_4 等
        m = re.search(r'ttb[_\s-]*?(\d+)\b', s, flags=re.IGNORECASE)
        return int(m.group(1)) if m else None
    df_sum['ttb'] = df_sum['sheet'].map(_pick_ttb)
    df_sum = df_sum.sort_values(['book','ttb'], na_position='last')

    # cols = ['ttb', 'avg_hops',  ]
    cols = ['ttb', 'avg_hops', 'std_val']


    print("\n=== SUMMARY (Average Switch Count) ===")
    print(df_sum[cols].to_string(index=False))

    ws = write2origin.ensure_sheet(TARGET_BOOK, OUT_SHEET, clear_existing=True, activate=True)
    ws.from_df(df_sum[cols])

    print(f"写入完成 -> [{TARGET_BOOK}] {OUT_SHEET}")
else:
    print("没有匹配到 pending_edges_* 的工作表。")


=== SUMMARY (Average Switch Count) ===
 ttb  avg_hops  std_val
  10 12.665293 1.864456
  20 12.715546 1.891702
  30 12.766340 1.916812
  40 12.816595 1.938528
  50 12.867215 1.959763
  60 12.918356 1.981540
  70 12.969066 2.002042
  80 13.018568 2.019297
  90 13.068239 2.037780
 100 13.115475 2.057206
 110 13.162664 2.075732
 120 13.210179 2.093577
 130 13.257975 2.110007
 140 13.305275 2.125505
写入完成 -> [Book2] avgspath_all 


In [6]:
import  importlib
importlib.reload(write2origin)

<module 'draw.pymatlab2.origin_function.write2origin' from 'C:\\usrspace\\mywork\\generic\\draw\\pymatlab2\\origin_function\\write2origin.py'>

here ,we solve the all_inter_edge


In [17]:
import draw.pymatlab2.origin_function.write2origin as write2origin
import re
TARGET_BOOK = 'Book2'


bk = op.find_book('w', TARGET_BOOK)
if bk is None:
    raise RuntimeError(f"找不到工作簿 '{TARGET_BOOK}'")

results = []

def _pick_ttb(s: str):
    m = re.search(r'ttb[_\s-]*?(\d+)\b', s, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

OUT_SHEET = 'avgspath_all_box'   # 去掉尾部空格，避免重名问题
long_rows = []                   # ← 新增：收集箱线图“长表”数据

for wks in bk:
    nm = wks.name
    if not nm.startswith('avgspath_g0_4_ttb'):
        continue

    # ===== 读取（Long Name 为表头）=====
    df = wks.to_df(head='L')
    if not {'step','avg_shortest_path'}.issubset(df.columns):
        print(f"跳过（缺列）: [{TARGET_BOOK}] {nm}")
        continue

    # ===== 规范化 =====
    df = df[['step','avg_shortest_path']].copy()
    df['step'] = pd.to_numeric(df['step'], errors='coerce')
    df['avg_shortest_path'] = pd.to_numeric(df['avg_shortest_path'], errors='coerce')
    df = df.dropna(subset=['step']).copy()
    df['step'] = df['step'].astype(int)
    df = df.sort_values('step')

    if df.empty:
        print(f"跳过（空表）: [{TARGET_BOOK}] {nm}")
        continue

    # === 计算均值&CV（相对波动） ===
    series = df['avg_shortest_path'].astype(float).dropna()
    avg_link = float(series.mean())
    std_val  = float(series.std(ddof=1)) if len(series) > 1 else 0.0
    cv = (std_val / avg_link) if avg_link != 0 else float('nan')

    ttb_val = _pick_ttb(nm)

    results.append({
        'book': TARGET_BOOK,
        'sheet': nm,
        'avg_hops': avg_link,
        'cv': cv,
    })

    # === 收集团队“长表”用于 Origin 画箱线图 ===
    if (ttb_val is not None) and (not series.empty):
        long_rows.append(pd.DataFrame({'ttb': ttb_val, 'asp': series.values}))

# ===== 汇总并写回 Origin =====
# ===== 汇总并写回 Origin =====
if results:
    df_sum = pd.DataFrame(results)
    df_sum['ttb'] = df_sum['sheet'].map(_pick_ttb)  # 若想稳妥保留也可
    df_sum = df_sum.sort_values(['book','ttb'], na_position='last')

    cols = ['ttb', 'avg_hops', 'cv']
    print("\n=== SUMMARY (Average Shortest Path & CV) ===")
    print(df_sum[cols].to_string(index=False))

    ws = write2origin.ensure_sheet(TARGET_BOOK, OUT_SHEET, clear_existing=True, activate=True)
    ws.from_df(df_sum[cols])
    print(f"写入完成 -> [{TARGET_BOOK}] {OUT_SHEET}")

    # === 写出箱线图用长表 ===
    if long_rows:
        df_long = pd.concat(long_rows, ignore_index=True)
        ws_long = write2origin.ensure_sheet(TARGET_BOOK, 'avgspath_long', clear_existing=True, activate=False)
        ws_long.from_df(df_long[['ttb','asp']])
        print("写入完成 -> [Book2] avgspath_long （列：ttb, asp）")
else:
    print("没有匹配到 pending_edges_* 的工作表。")



=== SUMMARY (Average Shortest Path & CV) ===
 ttb  avg_hops       cv
  10 12.665293 0.147210
  20 12.715546 0.148771
  30 12.766340 0.150146
  40 12.816595 0.151251
  50 12.867215 0.152307
  60 12.918356 0.153389
  70 12.969066 0.154371
  80 13.018568 0.155109
  90 13.068239 0.155934
 100 13.115475 0.156853
 110 13.162664 0.157698
 120 13.210179 0.158482
 130 13.257975 0.159150
 140 13.305275 0.159749
写入完成 -> [Book2] avgspath_all_box
写入完成 -> [Book2] avgspath_long （列：ttb, asp）


In [74]:
df

,time,pending_edges
0,0.0,548.0
1,1.0,548.0
2,2.0,548.0
3,3.0,548.0
4,4.0,548.0
...,...,...
22000,22000.0,597.0
22001,22001.0,597.0
22002,22002.0,597.0
22003,22003.0,597.0



=== SUMMARY (Average Switch Count) ===
 ttb  avarge_stable_time  max_stable_time  min_stable_time  qujian
  10           54.735000            168.0              1.0   400.0
  20           54.059259            156.0              1.0   405.0
  30           53.012107            146.0              1.0   413.0
  40           52.503597            136.0              2.0   417.0
  50           51.515294            126.0              1.0   425.0
  60           50.798144            116.0              1.0   431.0
  70           50.331034            106.0              1.0   435.0
  80           49.310811             96.0              1.0   444.0
  90           49.872437             90.0              1.0   439.0
 100           50.215596            100.0              1.0   436.0
 110           51.154206            110.0              1.0   428.0
 120           51.758865            120.0              1.0   423.0
 130           52.252983            130.0              1.0   419.0
 140           53.0121